# Divide & Conquer — Technical Reference

## Quick Index

| Technique | When to use | Section |
| :--- | :--- | :--- |
| Merge Sort | Sort + count inversions during sort | §1 |
| Quick Select | Kth smallest/largest in O(n) average | §2 |
| Binary Search (D&C view) | Eliminate half the search space | §3 |
| D&C on arrays | Max subarray, majority element | §4 |
| D&C on trees | Build tree from traversals, construct BST | §5 |

---
## When to Use

| Signal | Technique |
| :--- | :--- |
| "count inversions", "count smaller numbers after self" | Merge Sort |
| "kth largest/smallest" without full sort | Quick Select |
| "construct tree from inorder + preorder" | D&C on trees |
| "majority element" with O(1) space | D&C (Boyer-Moore or recursive) |
| "maximum subarray" O(n log n) | D&C (though Kadane's is O(n) — prefer Kadane's) |
| Sorted input, O(log n) required | Binary Search (D&C view) |

---
## §1 — Merge Sort & Count Inversions

Merge Sort splits the array in half, sorts each half recursively, then merges. The merge step is where counting inversions happens — when a right-half element is placed before a left-half element, it contributes inversions equal to the remaining left-half elements.

In [ ]:
# Merge Sort — O(n log n)
def merge_sort(nums):
    if len(nums) <= 1: return nums
    mid = len(nums) // 2
    left  = merge_sort(nums[:mid])
    right = merge_sort(nums[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]


# Count Inversions during merge — LC 315 / 493
# An inversion: right[j] comes before left[i] in sorted order
# When left[i] > right[j]: ALL remaining left elements form inversions with right[j]
def merge_count(nums):
    if len(nums) <= 1: return nums, 0
    mid = len(nums) // 2
    left,  left_inv  = merge_count(nums[:mid])
    right, right_inv = merge_count(nums[mid:])
    merged, split_inv = merge_and_count(left, right)
    return merged, left_inv + right_inv + split_inv

def merge_and_count(left, right):
    result = []
    inversions = 0
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            inversions += len(left) - i    # all remaining left elements > right[j]
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:], inversions

**Common mistakes:**
- Counting `len(left) - i` instead of `len(left) - i` when right element is placed — must count ALL remaining left elements, not just current
- Using `<` instead of `<=` in merge — causes instability and incorrect inversion count
- Forgetting to return both the sorted array and the count from each recursive call

Problems: LC 315, LC 493, LC 912

---
## §2 — Quick Select — Kth Smallest/Largest

Quick Select finds the kth smallest element in O(n) average time without fully sorting. Partition around a pivot — if pivot lands at index k, done. Otherwise recurse into only the relevant half. Worst case O(n²) with bad pivot choice — randomize pivot to avoid.

In [ ]:
import random

# Quick Select — Kth smallest (0-indexed k)
def quick_select(nums, k):
    if len(nums) == 1: return nums[0]

    pivot = random.choice(nums)         # randomize to avoid O(n²) worst case
    left  = [x for x in nums if x < pivot]
    mid   = [x for x in nums if x == pivot]
    right = [x for x in nums if x > pivot]

    if k < len(left):
        return quick_select(left, k)
    elif k < len(left) + len(mid):
        return pivot                    # pivot is the answer
    else:
        return quick_select(right, k - len(left) - len(mid))


# LC 215 — Kth Largest Element
# Kth largest = (n - k)th smallest (0-indexed)
def findKthLargest(nums, k):
    return quick_select(nums, len(nums) - k)


# In-place partition (Lomuto scheme) — O(1) space
def partition(nums, lo, hi):
    pivot = nums[hi]
    i = lo
    for j in range(lo, hi):
        if nums[j] <= pivot:
            nums[i], nums[j] = nums[j], nums[i]
            i += 1
    nums[i], nums[hi] = nums[hi], nums[i]
    return i                            # pivot's final index

**Common mistakes:**
- Using a fixed pivot (e.g. always `nums[-1]`) — causes O(n²) on sorted input; always randomize
- Off-by-one when converting kth largest to kth smallest — `len(nums) - k` for 0-indexed
- Forgetting the `mid` partition for duplicate values — without it, duplicates cause infinite recursion

Problems: LC 215, LC 347 (alternative to heap), LC 973

---
## §3 — Binary Search (D&C View)

Binary search is a specific application of divide & conquer where one half is always eliminated. The key insight: the search space must be **monotonic** — values transition from one state to another exactly once.

Full templates already in `cheatsheet_binary_search.ipynb`. This section covers only the D&C framing and the non-obvious applications.

In [ ]:
# LC 4 — Median of Two Sorted Arrays — O(log(min(m,n)))
# Binary search on partition point of smaller array
def findMedianSortedArrays(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1    # ensure nums1 is smaller
    m, n = len(nums1), len(nums2)
    lo, hi = 0, m

    while lo <= hi:
        i = (lo + hi) // 2             # partition point in nums1
        j = (m + n + 1) // 2 - i       # partition point in nums2

        max_left1  = nums1[i-1] if i > 0 else float('-inf')
        min_right1 = nums1[i]   if i < m else float('inf')
        max_left2  = nums2[j-1] if j > 0 else float('-inf')
        min_right2 = nums2[j]   if j < n else float('inf')

        if max_left1 <= min_right2 and max_left2 <= min_right1:
            if (m + n) % 2 == 1:
                return max(max_left1, max_left2)
            return (max(max_left1, max_left2) + min(min_right1, min_right2)) / 2
        elif max_left1 > min_right2:
            hi = i - 1
        else:
            lo = i + 1

**Common mistakes:**
- Not swapping arrays to ensure binary search on the smaller — always binary search on the shorter array
- Forgetting sentinel values (`-inf` / `inf`) for edge partitions — when `i = 0` or `i = m`, one side is empty

Problems: LC 4, LC 33, LC 81 — see `cheatsheet_binary_search.ipynb` for full templates

---
## §4 — D&C on Arrays

Some array problems are naturally solved by splitting at the midpoint and combining results from each half.

In [ ]:
# Maximum Subarray — D&C version (LC 53)
# Note: Kadane's O(n) is preferred — this is O(n log n)
# Useful when you need to understand the D&C pattern
def max_subarray_dc(nums, lo, hi):
    if lo == hi: return nums[lo]        # base case: single element

    mid = (lo + hi) // 2
    left_max  = max_subarray_dc(nums, lo, mid)
    right_max = max_subarray_dc(nums, mid + 1, hi)

    # Find max crossing subarray (must include mid and mid+1)
    left_sum = cur = 0
    for i in range(mid, lo - 1, -1):   # scan left from mid
        cur += nums[i]
        left_sum = max(left_sum, cur)

    right_sum = cur = 0
    for i in range(mid + 1, hi + 1):   # scan right from mid+1
        cur += nums[i]
        right_sum = max(right_sum, cur)

    cross_max = left_sum + right_sum
    return max(left_max, right_max, cross_max)


# Majority Element — D&C (LC 169)
# If element is majority in both halves, it's majority overall
def majority_element_dc(nums):
    def helper(lo, hi):
        if lo == hi: return nums[lo]
        mid = (lo + hi) // 2
        left  = helper(lo, mid)
        right = helper(mid + 1, hi)
        if left == right: return left
        # count which appears more in [lo..hi]
        left_count  = sum(1 for i in range(lo, hi+1) if nums[i] == left)
        right_count = sum(1 for i in range(lo, hi+1) if nums[i] == right)
        return left if left_count >= right_count else right

    return helper(0, len(nums) - 1)

**Common mistakes:**
- Forgetting the cross-boundary subarray in max subarray D&C — the optimal subarray may span both halves
- Using D&C for majority element when Boyer-Moore voting (O(n), O(1) space) is simpler — prefer Boyer-Moore

Problems: LC 53 (prefer Kadane's), LC 169 (prefer Boyer-Moore), LC 218

---
## §5 — D&C on Trees

Tree construction problems use D&C naturally — the root is identified, then left and right subtrees are built recursively from the remaining elements.

In [ ]:
# LC 105 — Construct Binary Tree from Preorder and Inorder
# Preorder[0] = root; find root in inorder to split left/right subtrees
def buildTree(preorder, inorder):
    if not preorder or not inorder: return None
    root_val = preorder[0]
    root = TreeNode(root_val)
    mid = inorder.index(root_val)       # index of root in inorder

    root.left  = buildTree(preorder[1:mid+1],  inorder[:mid])
    root.right = buildTree(preorder[mid+1:],   inorder[mid+1:])
    return root


# LC 108 — Convert Sorted Array to BST
# Always pick the middle element as root to keep tree balanced
def sortedArrayToBST(nums):
    if not nums: return None
    mid = len(nums) // 2
    root = TreeNode(nums[mid])
    root.left  = sortedArrayToBST(nums[:mid])
    root.right = sortedArrayToBST(nums[mid+1:])
    return root

**Common mistakes:**
- Using `inorder.index()` inside recursion without a hashmap — O(n) per call makes it O(n²); precompute `{val: index}` for O(1) lookup
- Off-by-one when slicing preorder — left subtree preorder is `preorder[1:mid+1]` (mid elements), not `preorder[1:mid]`
- In sorted array to BST: using `mid = (len-1) // 2` vs `mid = len // 2` — both work but produce different valid BSTs

Problems: LC 105, LC 106, LC 108, LC 109